[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-08-rag-retrieval.ipynb#scrollTo=a1b2c3d4)

---
# Day 8 · Retrieval-Augmented Generation — Build Your First RAG Chain
**certified-journeys / llm-engineering-certified** · Day 8 · Retrieval & Generation

> **Goal for today:** Build a complete RAG pipeline with LangChain LCEL — retriever, prompt, model, and output parser — then compare similarity vs. MMR retrieval and attach source citations to every answer.


In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb tiktoken


## Step 1 · What is RAG and why does it matter?

**Retrieval-Augmented Generation (RAG)** grounds an LLM's answer in real documents retrieved at query time — no fine-tuning needed.

| Component | Role | LangChain class |
|-----------|------|-----------------|
| Vector store | Stores embedded chunks | `Chroma` |
| Retriever | Fetches top-k relevant chunks | `VectorStore.as_retriever()` |
| Prompt | Injects retrieved context | `ChatPromptTemplate` |
| Model | Generates grounded answer | `ChatOpenAI` |
| Parser | Extracts plain text | `StrOutputParser` |

The LCEL pipe `retriever | format_docs | prompt | model | parser` wires all five together in one expression.


In [ ]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ------------------------------------------------------------------
# Set your OpenAI key (Colab: use Secrets or paste directly for demo)
# ------------------------------------------------------------------
os.environ.setdefault("OPENAI_API_KEY", "sk-...")

# ── Sample corpus ─────────────────────────────────────────────────
# In production this would be PDF / web content; we use inline text
# so the notebook runs without file I/O.
RAW_DOCS = [
    Document(
        page_content=(
            "LangChain Expression Language (LCEL) is a declarative way to compose chains. "
            "It uses the pipe operator | to connect runnables. LCEL supports streaming, "
            "async execution, and automatic parallel fan-out. Every LCEL chain is itself "
            "a Runnable, so chains can be nested arbitrarily."
        ),
        metadata={"source": "langchain-docs", "topic": "LCEL"},
    ),
    Document(
        page_content=(
            "Chroma is an open-source embedding database. It stores vectors alongside "
            "metadata and supports cosine, L2, and inner-product distance metrics. "
            "You can persist a Chroma collection to disk with persist_directory. "
            "LangChain wraps Chroma via langchain-community."
        ),
        metadata={"source": "chroma-docs", "topic": "vector stores"},
    ),
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) reduces hallucinations by supplying "
            "factual context at inference time. The retriever fetches the top-k chunks "
            "most similar to the query; those chunks are inserted into the prompt so the "
            "LLM can cite them directly rather than relying on parametric memory."
        ),
        metadata={"source": "rag-paper", "topic": "RAG"},
    ),
    Document(
        page_content=(
            "Maximal Marginal Relevance (MMR) balances relevance with diversity. "
            "It iteratively selects documents that are relevant to the query but "
            "dissimilar to already-selected documents. This prevents the retriever "
            "from returning near-duplicate chunks. Use search_type='mmr' in as_retriever()."
        ),
        metadata={"source": "mmr-paper", "topic": "MMR"},
    ),
    Document(
        page_content=(
            "OpenAI embeddings (text-embedding-3-small) produce 1536-dimensional vectors. "
            "Smaller models like text-embedding-3-small are cheaper and faster with only "
            "minor accuracy trade-offs. Always embed your documents and queries with the "
            "same model — mixing models will produce meaningless similarity scores."
        ),
        metadata={"source": "openai-docs", "topic": "embeddings"},
    ),
    Document(
        page_content=(
            "StrOutputParser is the simplest LangChain output parser. It extracts the "
            ".content string from an AIMessage. Use it as the final step in any LCEL "
            "chain that should return a plain string to the caller."
        ),
        metadata={"source": "langchain-docs", "topic": "output parsers"},
    ),
]

# Split into smaller chunks so retrieval is fine-grained
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(RAW_DOCS)
print(f"Created {len(chunks)} chunks from {len(RAW_DOCS)} source documents")


### What just happened?
- We loaded six synthetic documents covering LCEL, Chroma, RAG, MMR, embeddings, and parsers.
- `RecursiveCharacterTextSplitter` broke them into ≤300-character chunks with a 40-character overlap so no sentence is cut at a chunk boundary.
- **Key insight:** chunk size trades precision (small) for context completeness (large) — 300 chars is a safe default for dense technical text.
- Each chunk retains the original `metadata` dict, which we'll use later for source citations.


## Step 2 · Build a Chroma retriever with similarity search

We embed all chunks and store them in an **in-memory** Chroma collection, then expose a retriever with `as_retriever(search_type="similarity", search_kwargs={"k": 4})`.

```
Query ──► embed ──► cosine similarity ──► top-4 chunks
```

> **Tip from spec:** Always print retrieved docs alongside the answer during development. If retrieval is wrong, no prompt engineering will fix it.


In [ ]:
# Embed and index all chunks
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(chunks, embeddings)  # in-memory; no persist_directory

# Create a similarity retriever that returns 4 chunks per query
retriever_similarity = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Quick smoke test — always verify retrieval before building the chain
test_query = "What is MMR retrieval?"
test_docs = retriever_similarity.invoke(test_query)

print(f"Retrieved {len(test_docs)} chunks for: '{test_query}'")
print()
for i, doc in enumerate(test_docs, 1):
    src = doc.metadata.get("source", "unknown")
    print(f"[{i}] source={src}")
    print(f"     {doc.page_content[:120]}...")
    print()


### What just happened?
- `Chroma.from_documents` embedded every chunk and indexed the vectors.
- `retriever.invoke(query)` converts the query to a vector and returns the 4 nearest chunks by cosine similarity.
- **Key insight:** Verifying retrieval before building the full chain is the single most important debugging step in any RAG project — wrong retrieval = wrong answers no matter how good the prompt.
- The `source` metadata field is already attached to each returned `Document`, ready for citations.


## Step 3 · Construct the RAG chain with LCEL

The full chain follows the **Retrieve → Format → Prompt → Model → Parse** pattern:

```
question
  ├─► retriever          # fetch top-4 chunks
  └─► passthrough        # keep original question
        │
        ▼
      prompt             # system + human template
        │
        ▼
      ChatOpenAI         # generate grounded answer
        │
        ▼
      StrOutputParser    # extract .content string
```


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs: list) -> str:
    """Concatenate retrieved chunks into a single context string."""
    return "\n\n".join(d.page_content for d in docs)

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer the question using ONLY the context provided. "
     "If the context does not contain the answer, say 'Not covered in the provided context.'"),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# LCEL chain: fan-out question into retriever + passthrough, then prompt | model | parser
rag_chain = (
    {"context": retriever_similarity | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# ── Test with 5 questions ─────────────────────────────────────────
TEST_QUESTIONS = [
    "What does LCEL stand for and what operator does it use?",
    "How does MMR differ from plain similarity search?",
    "What embedding model should I use with Chroma?",
    "What is the role of StrOutputParser in a chain?",
    "Why does RAG reduce hallucinations?",
]

for q in TEST_QUESTIONS:
    answer = rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print("-" * 60)


### What just happened?
- The LCEL `{...}` dict at the start of the chain **fans out** the question: one branch runs retrieval, the other passes the question through unchanged.
- `format_docs` joins chunks with double newlines so the LLM sees them as separate paragraphs.
- **Key insight:** `temperature=0` is critical for RAG — we want deterministic, citation-grounded answers, not creative paraphrases.
- All five answers should be grounded in the retrieved text; if one says "Not covered", retrieval missed the relevant chunk.


## Step 4 · Print retrieved chunks alongside each answer

During development, always inspect **what the retriever actually returned** — not just the final answer. This lets you diagnose retrieval failures independently from generation failures.


In [ ]:
from langchain_core.runnables import RunnableParallel

# A chain that returns BOTH the answer and the retrieved docs
rag_with_sources = RunnableParallel(
    answer=rag_chain,
    sources=retriever_similarity,
)

debug_q = "How does MMR differ from plain similarity search?"
result = rag_with_sources.invoke(debug_q)

print("=" * 60)
print(f"QUESTION: {debug_q}")
print("=" * 60)
print(f"\nANSWER:\n{result['answer']}")
print("\nRETRIEVED CHUNKS:")
for i, doc in enumerate(result["sources"], 1):
    src = doc.metadata.get("source", "unknown")
    print(f"  [{i}] {src}: {doc.page_content[:100]}...")


### What just happened?
- `RunnableParallel` runs the answer chain and the retriever in parallel using the same input, then merges the outputs into a dict.
- **Key insight:** This debug pattern is a first-class workflow — keep it in your development loop and remove it only for production serving.
- If a retrieved chunk looks wrong (wrong topic, irrelevant source), fix chunking or embedding before touching the prompt.


## Step 5 · Switch to MMR retrieval and compare diversity

**Maximal Marginal Relevance** picks documents that are relevant to the query *and* dissimilar to each other. This matters when your corpus has near-duplicate chunks — similarity search returns them all; MMR spreads across topics.

| Parameter | Similarity | MMR |
|-----------|-----------|-----|
| `search_type` | `"similarity"` | `"mmr"` |
| Extra param | — | `fetch_k` (pool size before MMR re-ranking) |
| Diversity | Low (can return duplicates) | High |
| Cost | O(k) | O(fetch_k) embedding comparisons |


In [ ]:
# MMR retriever: fetch 10 candidates, return the 4 most diverse
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 10},
)

# Compare the two retrievers on a broad question that could match many chunks
broad_q = "Tell me about LangChain tools and retrieval"

sim_docs = retriever_similarity.invoke(broad_q)
mmr_docs = retriever_mmr.invoke(broad_q)

print("SIMILARITY retriever topics:", [d.metadata.get("topic") for d in sim_docs])
print("MMR         retriever topics:", [d.metadata.get("topic") for d in mmr_docs])
print()

# Count unique topics to quantify diversity
sim_topics = {d.metadata.get("topic") for d in sim_docs}
mmr_topics  = {d.metadata.get("topic") for d in mmr_docs}
print(f"Unique topics — similarity: {len(sim_topics)}, MMR: {len(mmr_topics)}")

# Build a second RAG chain with the MMR retriever
rag_chain_mmr = (
    {"context": retriever_mmr | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("\nMMR answer:\n", rag_chain_mmr.invoke(broad_q))


### What just happened?
- MMR fetches `fetch_k=10` candidates then re-ranks them, trading one extra scoring pass for higher topic diversity.
- **Key insight:** MMR shines when your corpus has many near-duplicate paragraphs (e.g., repeated boilerplate in legal docs) — similarity search would waste all k slots on variants of the same passage.
- For small, diverse corpora the improvement is minimal; for large corpora with repetition it can be dramatic.


## Step 6 · Add source metadata to every answer

Production RAG pipelines should show users *which document* an answer came from. We build a helper that:
1. Retrieves chunks and generates an answer (as before)
2. Deduplicates source references
3. Appends a `Sources:` footer to the answer


In [ ]:
def rag_with_citations(question: str, retriever=retriever_similarity) -> dict:
    """Run RAG and return answer + deduplicated source list."""
    # 1. Retrieve
    docs = retriever.invoke(question)

    # 2. Build context string
    context = format_docs(docs)

    # 3. Generate answer
    prompt_val = RAG_PROMPT.invoke({"context": context, "question": question})
    raw_answer = (llm | StrOutputParser()).invoke(prompt_val)

    # 4. Collect unique sources (preserve order of first appearance)
    seen = set()
    sources = []
    for doc in docs:
        src = doc.metadata.get("source", "unknown")
        if src not in seen:
            seen.add(src)
            sources.append({"source": src, "topic": doc.metadata.get("topic", "")})

    return {"answer": raw_answer, "sources": sources}


# Test all 5 questions with source citations
CITATION_QUESTIONS = [
    "What does LCEL stand for and what operator does it use?",
    "How does MMR differ from plain similarity search?",
    "What embedding model should I use with Chroma?",
    "What is the role of StrOutputParser?",
    "Why does RAG reduce hallucinations?",
]

for q in CITATION_QUESTIONS:
    r = rag_with_citations(q)
    citation_line = ", ".join(f"{s['source']} ({s['topic']})" for s in r["sources"])
    print(f"Q: {q}")
    print(f"A: {r['answer']}")
    print(f"   Sources: {citation_line}")
    print("-" * 70)


### What just happened?
- We stepped outside pure LCEL here deliberately to show how to access intermediate values (the `docs` list) when you need post-processing that LCEL does not expose natively.
- **Key insight:** The source list is deduplicated by `source` key — if two chunks came from the same document, we cite it only once.
- In a real product you would store chunk URLs or page numbers in metadata so citations link directly to the source.
- This pattern satisfies regulatory requirements (financial, legal, medical) where answer provenance is mandatory.


In [ ]:
# Challenge: Build a cited RAG chain that uses MMR retrieval
# and formats citations as a numbered list below the answer.
#
# Requirements:
#   1. Use retriever_mmr (already defined above)
#   2. The returned dict must have keys: 'answer', 'citations'
#   3. 'citations' is a list of strings like "[1] source-name — topic"
#   4. Test it on the question: "Explain MMR and embedding models"
#
# Scaffold:
def rag_mmr_cited(question: str) -> dict:
    # TODO: retrieve with retriever_mmr
    # TODO: generate answer using RAG_PROMPT + llm + StrOutputParser
    # TODO: build numbered citations list
    pass

# result = rag_mmr_cited("Explain MMR and embedding models")
# print(result["answer"])
# print("\nCitations:")
# for c in result["citations"]:
#     print(c)


---
## Day 8 key concepts recap

| Concept | What to remember |
|---------|------------------|
| RAG pipeline | retriever → format_docs → prompt → model → parser |
| `as_retriever(search_type="similarity")` | cosine nearest-neighbor, can return duplicates |
| `as_retriever(search_type="mmr")` | relevance + diversity; use `fetch_k` ≥ 2×k |
| `RunnableParallel` | fan-out; returns both answer and retrieved docs |
| Source citations | store `source` in `Document.metadata`; dedup by key |
| Debug rule | always print retrieved docs before debugging the LLM |

> **Tip:** Always print the retrieved docs alongside the answer during development — if retrieval is wrong, no prompt engineering will fix it.

---
## What's next
**Day 9** → Conversation Memory — Buffer, Summary, and RAG with History. We'll add stateful multi-turn memory to the RAG chain so follow-up questions use prior context.

Mark Day 8 complete in your [tracker](../index.html).
